# Agrupamiento de circuitos por UITI acumulado y numero de eventos

Cuaderno autosuficiente y de una sola figura. Carga `data/Indicadores_vano_v3.csv`, acumula por
`CIRCUITO` el UITI (`UITI_VANO`) y el numero de eventos, y segmenta los circuitos con K-Means en
**4 grupos** (`Bajo`, `Medio`, `Medio-Alto`, `Alto`) sobre el espacio ajustado.

La figura viene con un panel de cuatro controles independientes:

- **Desde / Hasta**: calendarios que acotan el periodo sobre el que se acumulan UITI y eventos.
  Por defecto, el rango completo de la base.
- **Log eje X** y **Log eje Y**: aplican `log10` a cada variable por separado, asi que se pueden
  combinar (solo x, solo y, las dos o ninguna).
- **Preproceso**: `minmax` o `z-score`.
- **Descargar etiquetas (CSV)**: baja la tabla de circuitos etiquetados con la configuracion
  que este a la vista. La ultima celda hace lo mismo desde Python, con `tabla_etiquetas()` /
  `guardar_etiquetas()`: mismo esquema y mismo orden, para que los dos caminos no diverjan.

La escala y el preproceso se aplican *antes* de correr K-Means, asi que cambian la particion, no
solo como se dibuja. Cada cambio actualiza a la vez el scatter, los contornos de membresia, las
densidades marginales y el conteo de circuitos por grupo.

Los grupos no se nombran por el id que devuelve K-Means (que es arbitrario) sino por el **ranking
de la mediana del UITI acumulado**: `Bajo` es el de menor mediana y `Alto` el de mayor.

> **Los calendarios ajustan a mes completo.** El menor grano precomputable es el mes: un rango
> diario exacto daria 16.471 combinaciones, imposible de embeber. La alternativa seria reimplementar
> K-Means en JavaScript, y entonces la particion que ves dejaria de ser la que calcula scikit-learn.
> El panel avisa cual es el rango efectivo cada vez que cambias una fecha.

> **Por que el panel sale de una celda de codigo y no de markdown.** JupyterLab sanitiza las celdas
> de markdown: `<input>` y `<button>` estan en su lista de tags permitidos, pero `<script>` no, y
> tampoco ningun atributo `on*`. Un calendario puesto en markdown se dibujaria y quedaria muerto.
> En la salida de una celda de codigo si corre JavaScript -- es el mismo mecanismo por el que se
> dibuja Plotly. Requiere que el cuaderno este *trusted*, igual que la figura.

In [1]:
# Descomentar solo si el entorno no tiene instaladas estas dependencias.
# %pip install pandas numpy scipy scikit-learn plotly

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler

NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
# Paleta Reds, de claro a oscuro, para que el color ya ordene los grupos por criticidad.
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
PREPROCESOS = {'minmax': MinMaxScaler, 'zscore': StandardScaler}
GRID_KDE = 64
SEMILLA = 42


# Sube desde el cwd hasta encontrar data/Indicadores_vano_v3.csv, para que el cuaderno
# funcione sin importar desde que directorio se ejecute (Jupyter local o Colab/Kaggle).
def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data/Indicadores_vano_v3.csv').exists():
            return candidate
    raise FileNotFoundError('No se encontro data/Indicadores_vano_v3.csv subiendo desde el cwd')


REPO_ROOT = find_repo_root()

# Solo 3 columnas: el CSV completo trae ~270 columnas climaticas por evento.
df = pd.read_csv(REPO_ROOT / 'data' / 'Indicadores_vano_v3.csv',
                 usecols=['CIRCUITO', 'UITI_VANO', 'FECHA'])
df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')
df['UITI_VANO'] = pd.to_numeric(df['UITI_VANO'], errors='coerce').fillna(0.0)
df['MES'] = df['FECHA'].dt.to_period('M')

MESES = sorted(df['MES'].unique())
# Todos los pares (desde, hasta) con desde <= hasta: el calendario ajusta a uno de estos.
RANGOS = [(i, j) for i in range(len(MESES)) for j in range(i, len(MESES))]
IDX_RANGO_COMPLETO = RANGOS.index((0, len(MESES) - 1))
# log del eje x y del eje y son independientes: 2 x 2 x 2 preprocesos = 8 espacios.
ESPACIOS = [(lx, ly, prep)
            for lx in (False, True) for ly in (False, True)
            for prep in ('minmax', 'zscore')]
IDX_ESPACIO_DEFECTO = ESPACIOS.index((False, True, 'minmax'))

print(f'{len(df):,} eventos | {df["CIRCUITO"].nunique()} circuitos | '
      f'{df["FECHA"].min():%Y-%m-%d} a {df["FECHA"].max():%Y-%m-%d}')
print(f'{len(RANGOS)} rangos x {len(ESPACIOS)} espacios = {len(RANGOS) * len(ESPACIOS)} combinaciones')

159,470 eventos | 208 circuitos | 2025-11-01 a 2026-04-30
21 rangos x 8 espacios = 168 combinaciones


In [3]:
def tabla_circuitos(desde, hasta):
    """UITI acumulado y numero de eventos por circuito, restringido a los meses [desde, hasta]."""
    sub = df[(df['MES'] >= MESES[desde]) & (df['MES'] <= MESES[hasta])]
    return (
        sub.groupby('CIRCUITO')
        # Un evento es una FECHA distinta, no una fila: una misma salida golpea muchos
        # vanos y genera muchas filas. Es la definicion que usa el agrupamiento de
        # circuitos del reporte (`count_unique_event_dates`, en
        # `compute_circuit_criticality_groups`); contar filas daba una mediana de 468
        # eventos por circuito contra los 18 reales, hasta 111 veces mas.
        .agg(uiti_acumulado=('UITI_VANO', 'sum'), num_eventos=('FECHA', 'nunique'))
        .reset_index()
    )


def aplicar_log(X, logs):
    """log10 columna por columna, segun que ejes lo tengan activado."""
    # Seguro en esta base: el minimo por circuito es 2 eventos y 0.25 de UITI, nunca 0.
    V = np.array(X, dtype=float, copy=True)
    for c, activo in enumerate(logs):
        if activo:
            V[:, c] = np.log10(V[:, c])
    return V


def agrupar(tabla, logs, prep):
    """K-Means a 4 grupos sobre el espacio ajustado.

    Devuelve la tabla etiquetada y la geometria de la particion (centroides y parametros
    del escalador), que es lo que necesitan los contornos de membresia.
    """
    X = aplicar_log(tabla[['num_eventos', 'uiti_acumulado']].to_numpy(dtype=float), logs)

    escalador = PREPROCESOS[prep]().fit(X)
    modelo = KMeans(n_clusters=4, random_state=SEMILLA, n_init=10).fit(escalador.transform(X))
    tabla = tabla.assign(_cluster=modelo.labels_)

    # El id que devuelve K-Means es arbitrario: el nombre del grupo se asigna por el
    # ranking de la MEDIANA del UITI acumulado, de menor a mayor.
    orden = tabla.groupby('_cluster')['uiti_acumulado'].median().sort_values().index.tolist()
    tabla['grupo'] = tabla['_cluster'].map({c: i for i, c in enumerate(orden)})

    # Todo escalador se reduce a (v - offset) / scale, asi el JS aplica uno solo.
    if prep == 'minmax':
        offset, scale = escalador.data_min_, escalador.data_range_
    else:
        offset, scale = escalador.mean_, escalador.scale_

    geometria = {
        'logs': [bool(logs[0]), bool(logs[1])],
        'offset': np.round(offset, 6).tolist(),
        'scale': np.round(scale, 6).tolist(),
        # Centroides reordenados al mismo indice de grupo que las etiquetas.
        'centroides': np.round(modelo.cluster_centers_[orden], 6).tolist(),
    }
    return tabla.drop(columns='_cluster'), geometria


def membresia(X_display, geometria):
    """Grupo de cada punto por centroide mas cercano; replica en numpy lo que hace el JS."""
    Z = ((aplicar_log(X_display, geometria['logs']) - np.array(geometria['offset']))
         / np.array(geometria['scale']))
    d = ((Z[:, None, :] - np.array(geometria['centroides'])[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(axis=1)


def curva_kde(valores, log):
    """Densidad de una variable en el espacio en que corrio K-Means, devuelta en unidades originales."""
    valores = np.asarray(valores, dtype=float)
    if valores.size < 3 or np.allclose(valores, valores[0]):
        return [], []
    base = np.log10(valores) if log else valores
    grid = np.linspace(base.min(), base.max(), GRID_KDE)
    densidad = gaussian_kde(base)(grid)
    # Se redondea antes de serializar: el cuaderno embebe las 168 combinaciones y los
    # decimales de mas solo inflan su tamano.
    return (np.round(10.0 ** grid if log else grid, 4).tolist(),
            np.round(densidad, 6).tolist())

In [4]:
# El JavaScript del panel no ajusta ningun modelo: solo intercambia datos ya agrupados y
# evalua "centroide mas cercano" sobre una grilla para dibujar los contornos. Cada
# combinacion se resuelve aca, con scikit-learn, y viaja embebida en la salida.
# K-Means se ajusta UNA sola vez por espacio, sobre la ventana temporal completa. Los
# centroides y el escalador quedan fijos: cambiar el rango de fechas ya no redefine los
# grupos, solo mueve los circuitos dentro de una particion que no se mueve. Antes cada
# rango reajustaba las fronteras, y "Alto" no significaba lo mismo de un rango a otro.
TABLA_COMPLETA = tabla_circuitos(0, len(MESES) - 1)
GEOMETRIA_FIJA = {}
for e, (log_x, log_y, prep) in enumerate(ESPACIOS):
    _agrupada_full, _geo = agrupar(TABLA_COMPLETA, (log_x, log_y), prep)
    # La regla de centroide mas cercano tiene que reproducir las etiquetas del ajuste,
    # porque es la unica que se usa despues para cualquier otro rango.
    assert np.array_equal(
        membresia(TABLA_COMPLETA[['num_eventos', 'uiti_acumulado']].to_numpy(float), _geo),
        _agrupada_full['grupo'].to_numpy(),
    ), f'la regla de centroide mas cercano no reproduce el ajuste del espacio {e}'
    GEOMETRIA_FIJA[e] = _geo

# Extension FIJA, tomada de la ventana completa: es la misma sobre la que se ajustaron los
# centroides. Con ella los ejes y la grilla del contorno no se mueven al cambiar el rango,
# asi dos rangos distintos se pueden comparar mirando donde caen los puntos.
EXTENSION_FIJA = [
    float(TABLA_COMPLETA['num_eventos'].min()), float(TABLA_COMPLETA['num_eventos'].max()),
    float(round(TABLA_COMPLETA['uiti_acumulado'].min(), 4)),
    float(round(TABLA_COMPLETA['uiti_acumulado'].max(), 4)),
]

COMBINACIONES = {}
for r, (desde, hasta) in enumerate(RANGOS):
    tabla = tabla_circuitos(desde, hasta)
    for e, (log_x, log_y, prep) in enumerate(ESPACIOS):
        geometria = GEOMETRIA_FIJA[e]
        # Membresia del rango elegido contra los centroides fijos: no se reajusta nada.
        agrupada = tabla.assign(grupo=membresia(
            tabla[['num_eventos', 'uiti_acumulado']].to_numpy(float), geometria))

        bloque = []
        for g in range(4):
            sel = agrupada[agrupada['grupo'] == g]
            kde_x, dens_x = curva_kde(sel['num_eventos'], log_x)
            kde_y, dens_y = curva_kde(sel['uiti_acumulado'], log_y)
            bloque.append({
                'x': sel['num_eventos'].astype(int).tolist(),
                'y': np.round(sel['uiti_acumulado'], 2).tolist(),
                'circuitos': sel['CIRCUITO'].tolist(),
                'kde_x': kde_x, 'dens_x': dens_x,
                'kde_y': kde_y, 'dens_y': dens_y,
            })

        COMBINACIONES[f'{r}|{e}'] = {
            'grupos': bloque,
            'geometria': geometria,
        }

# El nombre del grupo solo significa algo si el ranking por mediana es estrictamente creciente.
# Con centroides fijos el orden por mediana esta garantizado solo en la ventana completa:
# en un rango corto un grupo puede quedar con pocos circuitos y cruzarse con el vecino.
# Se cuenta y se informa en vez de asumirlo.
rotas = [
    clave for clave, combo in COMBINACIONES.items()
    if not all(np.median(combo['grupos'][g]['y']) < np.median(combo['grupos'][g + 1]['y'])
               for g in range(3))
]
print(f'{len(COMBINACIONES)} combinaciones | K-Means ajustado {len(ESPACIOS)} veces '
      f'(una por espacio, sobre la ventana completa)')
print(f'rangos donde el orden por mediana no se sostiene: {len(rotas)} de {len(COMBINACIONES)}')

168 combinaciones | K-Means ajustado 8 veces (una por espacio, sobre la ventana completa)
rangos donde el orden por mediana no se sostiene: 51 de 168


/Users/andresalvarez/Documents/chec-local-uiti-vano-interpreter/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/andresalvarez/Documents/chec-local-uiti-vano-interpreter/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [5]:
# La figura tiene 22 trazas fijas y siempre son esas 22: 1 contorno de membresia,
# 4 scatter, 4 KDE del eje x, 4 KDE del eje y, 1 barra de conteos y 8 violines (4 del UITI
# acumulado + 4 del numero de eventos). El panel no crea ni destruye trazas, solo les
# reescribe los datos; los violines reusan los mismos arrays por grupo que el scatter, asi
# que no agregan nada al payload.
fig = make_subplots(
    rows=5, cols=2,
    row_heights=[0.13, 0.40, 0.15, 0.16, 0.16], column_widths=[0.78, 0.22],
    horizontal_spacing=0.02, vertical_spacing=0.07,
)

inicial = COMBINACIONES[f'{IDX_RANGO_COMPLETO}|{IDX_ESPACIO_DEFECTO}']
grupos_ini = inicial['grupos']

# Escala discreta de 4 escalones: cada banda del contorno toma el color de su grupo.
ESCALA_CONTORNO = []
for g, color in enumerate(COLORES_GRUPOS):
    ESCALA_CONTORNO.append([g / 4.0, color])
    ESCALA_CONTORNO.append([(g + 1) / 4.0, color])

fig.add_trace(go.Contour(                                       # traza 0: membresia
    z=[[0, 0], [0, 0]], x=[0, 1], y=[0, 1],
    colorscale=ESCALA_CONTORNO, zmin=-0.5, zmax=3.5, showscale=False,
    opacity=0.28, hoverinfo='skip', line=dict(width=1.2, color='rgba(120,20,20,0.6)'),
    contours=dict(start=-0.5, end=3.5, size=1, coloring='fill'),
    name='Membresia', showlegend=False,
), row=2, col=1)
for g in range(4):                                              # trazas 1-4: scatter
    fig.add_trace(go.Scattergl(
        x=grupos_ini[g]['x'], y=grupos_ini[g]['y'], mode='markers', name=NOMBRES_GRUPOS[g],
        legendgroup=NOMBRES_GRUPOS[g],
        marker=dict(size=9, color=COLORES_GRUPOS[g],
                    line=dict(width=0.5, color='rgba(60,10,10,0.6)')),
        customdata=grupos_ini[g]['circuitos'],
        # El grupo va fijo en la plantilla, no en customdata: la traza g siempre es el
        # grupo g, lo unico que cambia con los filtros es que circuitos caen adentro.
        hovertemplate=(f'<b>%{{customdata}}</b><br>Grupo: <b>{NOMBRES_GRUPOS[g]}</b>'
                       '<br>Eventos: %{x:,}<br>UITI acumulado: %{y:,.1f}<extra></extra>'),
    ), row=2, col=1)
for g in range(4):                                              # trazas 5-8: KDE eje x
    fig.add_trace(go.Scatter(
        x=grupos_ini[g]['kde_x'], y=grupos_ini[g]['dens_x'], mode='lines', fill='tozeroy',
        name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g], showlegend=False,
        line=dict(width=1.2, color=COLORES_GRUPOS[g]),
        fillcolor=COLORES_GRUPOS[g].replace('rgb', 'rgba').replace(')', ',0.25)'),
        hoverinfo='skip',
    ), row=1, col=1)
for g in range(4):                                              # trazas 9-12: KDE eje y
    fig.add_trace(go.Scatter(
        x=grupos_ini[g]['dens_y'], y=grupos_ini[g]['kde_y'], mode='lines', fill='tozerox',
        name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g], showlegend=False,
        line=dict(width=1.2, color=COLORES_GRUPOS[g]),
        fillcolor=COLORES_GRUPOS[g].replace('rgb', 'rgba').replace(')', ',0.25)'),
        hoverinfo='skip',
    ), row=2, col=2)
conteos_ini = [len(grupos_ini[g]['x']) for g in range(4)]
fig.add_trace(go.Bar(                                           # traza 13: conteo por grupo
    x=NOMBRES_GRUPOS, y=conteos_ini, text=conteos_ini, textposition='outside',
    marker=dict(color=COLORES_GRUPOS, line=dict(width=0.5, color='rgba(60,10,10,0.6)')),
    showlegend=False, hovertemplate='%{x}: %{y} circuitos<extra></extra>', cliponaxis=False,
), row=3, col=1)
# Trazas 14-17 y 18-21: distribucion completa por grupo de cada variable. `box_visible`
# deja la mediana a la vista, que es exactamente el criterio con que se nombran los grupos.
for fila, (clave, etiqueta) in enumerate([('y', 'UITI acumulado'), ('x', 'Numero de eventos')]):
    for g in range(4):
        fig.add_trace(go.Violin(
            x=[NOMBRES_GRUPOS[g]] * len(grupos_ini[g][clave]), y=grupos_ini[g][clave],
            name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g], showlegend=False,
            line=dict(color='rgba(90,15,20,0.85)', width=1),
            fillcolor=COLORES_GRUPOS[g], opacity=0.85,
            box_visible=True, meanline_visible=False, points=False, spanmode='hard',
            hovertemplate=f'%{{x}} -- {etiqueta}: %{{y:,.1f}}<extra></extra>',
        ), row=4 + fila, col=1)

# Traza 22: el porcentaje va DENTRO de la barra. Una traza de barras tiene un solo
# `text`/`textposition`, asi que el conteo de afuera y el porcentaje de adentro no pueden
# salir de la misma; este texto se posiciona a media altura de cada barra.
fig.add_trace(go.Scatter(
    x=NOMBRES_GRUPOS, y=[c / 2 for c in conteos_ini], mode='text',
    text=[f'{100 * c / max(sum(conteos_ini), 1):.1f}%' for c in conteos_ini],
    # Color por punto: sobre las dos barras oscuras un texto oscuro no se lee. Los grupos
    # estan ordenados de claro a oscuro, asi que el corte es fijo.
    textposition='middle center',
    textfont=dict(size=11, color=['rgb(40,10,12)', 'rgb(40,10,12)', 'white', 'white']),
    showlegend=False, hoverinfo='skip',
), row=3, col=1)

fig.update_layout(
    title=dict(
        text='Agrupamiento de circuitos por UITI acumulado y numero de eventos'
             '<br><sup>K-Means (k=4); los grupos se nombran por el ranking de la mediana '
             'del UITI acumulado</sup>',
        x=0.5, xanchor='center', yref='container', y=0.96, yanchor='top',
    ),
    legend=dict(title_text='', orientation='h', x=0.5, xanchor='center', y=1.02, yanchor='bottom'),
    margin=dict(t=110, r=30, b=60, l=90),
    height=1260, width=820, template='plotly_white', bargap=0.45,
)
# El KDE superior comparte el eje x del scatter y el KDE derecho su eje y; el de barras es
# categorico y queda independiente a proposito.
fig.update_xaxes(matches='x3', row=1, col=1)
fig.update_yaxes(matches='y3', row=2, col=2)
fig.update_xaxes(title_text='Numero de eventos', row=2, col=1)
fig.update_yaxes(title_text='UITI acumulado', row=2, col=1)
# La densidad se lee por forma, no por valor: sus ticks solo agregan ruido (y sobre el eje del
# UITI salen en notacion micro, porque la densidad esta en unidades de 1/UITI).
fig.update_yaxes(title_text='Densidad', showticklabels=False, row=1, col=1)
fig.update_xaxes(title_text='Densidad', showticklabels=False, row=2, col=2)
fig.update_yaxes(title_text='Circuitos', rangemode='tozero', row=3, col=1)
fig.update_xaxes(visible=False, row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)
fig.update_yaxes(title_text='UITI acumulado', row=4, col=1)
fig.update_yaxes(title_text='Numero de eventos', row=5, col=1)
for fila_vacia in (3, 4, 5):
    fig.update_xaxes(visible=False, row=fila_vacia, col=2)
    fig.update_yaxes(visible=False, row=fila_vacia, col=2)

# El print cierra la celda a proposito: si terminara en un update_*(), Jupyter mostraria
# la Figure devuelta y quedarian dos figuras, una de ellas sin panel de control.
print(f'{len(fig.data)} trazas: 1 contorno + 4 scatter + 8 KDE + 1 barras + 8 violines')

23 trazas: 1 contorno + 4 scatter + 8 KDE + 1 barras + 8 violines


In [6]:
DIV_FIGURA = 'agrupamiento-circuitos'
PRIMER_DIA = f'{MESES[0]}-01'
ULTIMO_DIA = str(df['FECHA'].max().date())
RESOLUCION_CONTORNO = 90

# Todo lo que el JS necesita, resuelto en Python: no ajusta modelos ni recalcula agregados.
CONTEXTO = {
    'div': DIV_FIGURA,
    'meses': [str(m) for m in MESES],
    # Ultimo dia real de cada mes: el CSV declara el periodo cerrado, no el mes suelto.
    'finMes': [str(m.to_timestamp(how='end').date()) for m in MESES],
    'rangos': RANGOS,
    'espacios': [[bool(lx), bool(ly), prep] for lx, ly, prep in ESPACIOS],
    'grupos': NOMBRES_GRUPOS,
    'resolucion': RESOLUCION_CONTORNO,
    'combinaciones': COMBINACIONES,
    'extension': EXTENSION_FIJA,
}

PANEL_HTML = f'''
<style>
  .panel-agrup {{
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif; font-size: 13px;
    display: flex; flex-wrap: wrap; gap: 18px; align-items: flex-end;
    max-width: 820px; margin: 0 0 6px 0; padding: 12px 14px;
    border: 1px solid #e4c4c0; border-left: 4px solid rgb(203,24,29);
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b;
  }}
  .panel-agrup label {{ display: block; font-weight: 600; margin-bottom: 4px; }}
  .panel-agrup input[type="date"], .panel-agrup select {{
    font: inherit; padding: 4px 6px; border: 1px solid #c9a9a5;
    border-radius: 4px; background: #fff; color: #2b2b2b;
  }}
  .panel-agrup .chk {{ font-weight: 600; display: flex; align-items: center; gap: 6px; }}
  .panel-agrup .chk input {{ margin: 0; }}
  .panel-agrup .grupo-chk {{ display: flex; flex-direction: column; gap: 6px; }}
  .panel-agrup button {{
    font: inherit; font-weight: 600; padding: 6px 12px; cursor: pointer;
    border: 1px solid rgb(203,24,29); border-radius: 4px;
    background: rgb(203,24,29); color: #fff;
  }}
  .panel-agrup button:hover {{ background: rgb(165,15,21); }}
  .panel-aviso {{
    flex-basis: 100%; font-size: 12px; color: #7a5c58; margin: 0; font-weight: 400;
  }}
</style>
<div class="panel-agrup">
  <div><label for="ag-desde">Desde</label>
       <input type="date" id="ag-desde" min="{PRIMER_DIA}" max="{ULTIMO_DIA}" value="{PRIMER_DIA}"></div>
  <div><label for="ag-hasta">Hasta</label>
       <input type="date" id="ag-hasta" min="{PRIMER_DIA}" max="{ULTIMO_DIA}" value="{ULTIMO_DIA}"></div>
  <div class="grupo-chk">
    <label class="chk"><input type="checkbox" id="ag-logx"> Log eje X (eventos)</label>
    <label class="chk"><input type="checkbox" id="ag-logy" checked> Log eje Y (UITI)</label>
  </div>
  <div><label for="ag-prep">Preproceso</label>
       <select id="ag-prep"><option value="minmax">minmax</option>
                            <option value="zscore">z-score</option></select></div>
  <div><button type="button" id="ag-csv">Descargar etiquetas (CSV)</button></div>
  <p class="panel-aviso" id="ag-aviso"></p>
</div>
'''

PANEL_JS = '''
<script type="text/javascript">
(function () {
  var CTX = %s;
  var d = document;

  function idxMes(valor) {
    // El calendario ajusta a mes completo: se toma el mes de la fecha elegida y se
    // acota al rango disponible en la base.
    var i = CTX.meses.indexOf((valor || '').slice(0, 7));
    return i < 0 ? null : i;
  }

  function idxRango(desde, hasta) {
    for (var i = 0; i < CTX.rangos.length; i++) {
      if (CTX.rangos[i][0] === desde && CTX.rangos[i][1] === hasta) return i;
    }
    return null;
  }

  function ejeGrilla(min, max, n, log) {
    // En logaritmica la grilla se reparte geometricamente, para que quede pareja en pantalla.
    var out = [], lo = log ? Math.log10(min) : min, hi = log ? Math.log10(max) : max;
    var paso = (hi - lo) / (n - 1);
    for (var i = 0; i < n; i++) {
      var v = lo + i * paso;
      out.push(log ? Math.pow(10, v) : v);
    }
    return out;
  }

  function contorno(combo) {
    // Membresia por centroide mas cercano en el espacio ajustado: la misma regla que
    // scikit-learn usa en predict(), verificada contra sus etiquetas del lado de Python.
    var geo = combo.geometria, ext = CTX.extension, n = CTX.resolucion;
    var lx = geo.logs[0], ly = geo.logs[1];
    var gx = ejeGrilla(ext[0], ext[1], n, lx);
    var gy = ejeGrilla(ext[2], ext[3], n, ly);
    var cen = geo.centroides, off = geo.offset, esc = geo.scale;
    var z = [];
    for (var j = 0; j < n; j++) {
      var ty = ((ly ? Math.log10(gy[j]) : gy[j]) - off[1]) / esc[1];
      var fila = [];
      for (var i = 0; i < n; i++) {
        var tx = ((lx ? Math.log10(gx[i]) : gx[i]) - off[0]) / esc[0];
        var mejor = 0, dmin = Infinity;
        for (var c = 0; c < cen.length; c++) {
          var a = tx - cen[c][0], b = ty - cen[c][1], dd = a * a + b * b;
          if (dd < dmin) { dmin = dd; mejor = c; }
        }
        fila.push(mejor);
      }
      z.push(fila);
    }
    return {z: z, x: gx, y: gy};
  }

  var ESTADO = null;   // configuracion vigente, para que el boton exporte lo que se ve

  function csvActual() {
    // Mismo esquema y mismo orden que tabla_etiquetas() en Python: si los dos caminos
    // no coincidieran, el CSV del boton y el del kernel contarian historias distintas.
    var s = ESTADO, filas = [];
    for (var g = 0; g < 4; g++) {
      var b = s.bloque[g];
      for (var i = 0; i < b.circuitos.length; i++) {
        filas.push({c: b.circuitos[i], e: CTX.grupos[g], n: b.x[i], u: b.y[i]});
      }
    }
    filas.sort(function (p, q) { return q.u - p.u; });   // UITI acumulado descendente

    var desde = CTX.meses[s.a] + '-01';
    var hasta = CTX.finMes[s.b];
    var cab = ['desde', 'hasta', 'circuito', 'etiqueta', 'num_eventos', 'uiti_acumulado'];
    var out = [cab.join(',')];
    filas.forEach(function (f) {
      out.push([desde, hasta, '"' + f.c + '"', f.e, f.n, f.u].join(','));
    });
    return {texto: out.join('\\n') + '\\n',
            nombre: 'etiquetas_circuitos_' + CTX.meses[s.a] + '_' + CTX.meses[s.b] +
                    '_x' + (s.logx ? 'log' : 'lin') + '_y' + (s.logy ? 'log' : 'lin') +
                    '_' + s.prep + '.csv'};
  }

  function descargar() {
    if (!ESTADO) { return; }
    var csv = csvActual();
    var url = URL.createObjectURL(new Blob([csv.texto], {type: 'text/csv;charset=utf-8;'}));
    var a = d.createElement('a');
    a.href = url; a.download = csv.nombre;
    d.body.appendChild(a); a.click(); d.body.removeChild(a);
    setTimeout(function () { URL.revokeObjectURL(url); }, 0);
  }

  function aplicar() {
    var gd = d.getElementById(CTX.div);
    if (!gd || !gd._fullLayout) { return setTimeout(aplicar, 120); }

    var a = idxMes(d.getElementById('ag-desde').value);
    var b = idxMes(d.getElementById('ag-hasta').value);
    if (a === null) a = 0;
    if (b === null) b = CTX.meses.length - 1;
    if (a > b) { var t = a; a = b; b = t; }   // fechas invertidas: se ordenan solas

    var logx = d.getElementById('ag-logx').checked;
    var logy = d.getElementById('ag-logy').checked;
    var prep = d.getElementById('ag-prep').value;
    var e = 0;
    for (var i = 0; i < CTX.espacios.length; i++) {
      var esp = CTX.espacios[i];
      if (esp[0] === logx && esp[1] === logy && esp[2] === prep) { e = i; break; }
    }

    var combo = CTX.combinaciones[idxRango(a, b) + '|' + e];
    if (!combo) { return; }
    var bloque = combo.grupos;
    ESTADO = {a: a, b: b, logx: logx, logy: logy, prep: prep, bloque: bloque};

    var sx = [], sy = [], scd = [], kx = [], ky = [], conteos = [];
    for (var g = 0; g < 4; g++) {
      sx.push(bloque[g].x); sy.push(bloque[g].y); scd.push(bloque[g].circuitos);
      conteos.push(bloque[g].x.length);
    }
    for (var g = 0; g < 4; g++) { kx.push(bloque[g].kde_x); ky.push(bloque[g].dens_x); }
    for (var g = 0; g < 4; g++) { kx.push(bloque[g].dens_y); ky.push(bloque[g].kde_y); }

    var ct = contorno(combo);
    Plotly.restyle(gd, {z: [ct.z], x: [ct.x], y: [ct.y]}, [0]);
    Plotly.restyle(gd, {x: sx, y: sy, customdata: scd}, [1, 2, 3, 4]);
    Plotly.restyle(gd, {x: kx, y: ky}, [5, 6, 7, 8, 9, 10, 11, 12]);
        // El porcentaje va DENTRO de la barra solo si la barra es lo bastante alta. En una
    // barra muy baja el texto de media altura cae sobre el eje y se encima con el nombre
    // del grupo; en ese caso se pega al conteo de afuera, que es donde si hay sitio.
    var totalC = conteos[0] + conteos[1] + conteos[2] + conteos[3];
    var maxC = Math.max.apply(null, conteos) || 1;
    var pctTxt = conteos.map(function (c) {
      // Simbolo duplicado: el bloque se arma con formateo de cadena.
      return totalC ? (100 * c / totalC).toFixed(1) + '%%' : '';
    });
    var bajo = conteos.map(function (c) { return c / maxC < 0.12; });
    Plotly.restyle(gd, {y: [conteos], text: [conteos.map(
      function (c, i) { return bajo[i] ? c + '  ' + pctTxt[i] : String(c); })]},
      [13]);
    Plotly.restyle(gd, {
      y: [conteos.map(function (c) { return c / 2; })],
      text: [pctTxt.map(function (p, i) { return bajo[i] ? '' : p; })],
    }, [22]);
    // El porcentaje se recalcula sobre el total de circuitos de la combinacion vigente,
    // no sobre los 208: si el rango deja circuitos sin eventos, el reparto es sobre los
    // que efectivamente entraron.
    // Los violines reciben los mismos arrays del scatter: el UITI acumulado va contra el
    // eje y del grupo y el numero de eventos contra su eje x.
    var vx = [], vy = [];
    for (var g = 0; g < 4; g++) {
      vx.push(bloque[g].y.map(function () { return CTX.grupos[g]; })); vy.push(bloque[g].y);
    }
    Plotly.restyle(gd, {x: vx, y: vy}, [14, 15, 16, 17]);
    vx = []; vy = [];
    for (var g = 0; g < 4; g++) {
      vx.push(bloque[g].x.map(function () { return CTX.grupos[g]; })); vy.push(bloque[g].x);
    }
    Plotly.restyle(gd, {x: vx, y: vy}, [18, 19, 20, 21]);
    // xaxis/xaxis3 son el eje x del KDE superior y del scatter; yaxis3/yaxis4 el eje y del
    // scatter y del KDE derecho. Cada eje toma su propio log. El de barras no se toca.
    var tx = logx ? 'log' : 'linear', ty = logy ? 'log' : 'linear';
    // Limites FIJOS, de la ventana completa. Sin esto cada rango reencuadra los ejes y un
    // circuito parece moverse cuando lo unico que cambio fue la escala. En log Plotly
    // espera el rango YA en log10.
    var ex = CTX.extension;
    function lim(lo, hi, log) {
      return log ? [Math.log10(lo * 0.85), Math.log10(hi * 1.15)]
                 : [0, hi * 1.05];
    }
    var rx = lim(ex[0], ex[1], logx), ry = lim(ex[2], ex[3], logy);
    // yaxis7 es el violin del UITI y yaxis9 el de eventos: cada uno sigue el log de su
    // propia variable, no el del eje en que esta dibujado.
    Plotly.relayout(gd, {'xaxis.type': tx, 'xaxis3.type': tx,
                         'yaxis3.type': ty, 'yaxis4.type': ty,
                         'yaxis7.type': ty, 'yaxis9.type': tx,
                         // xaxis3/yaxis3 son los del scatter y xaxis/yaxis4 los de sus
                         // marginales: los cuatro comparten los limites fijos.
                         'xaxis.range': rx, 'xaxis3.range': rx,
                         'yaxis3.range': ry, 'yaxis4.range': ry});

    var n = conteos.reduce(function (s, v) { return s + v; }, 0);
    d.getElementById('ag-aviso').textContent =
      'Rango efectivo: ' + CTX.meses[a] + ' a ' + CTX.meses[b] +
      ' (ajustado a meses completos) \\u2014 ' + n + ' circuitos con eventos en el periodo.';
  }

  ['ag-desde', 'ag-hasta', 'ag-logx', 'ag-logy', 'ag-prep'].forEach(function (id) {
    var el = d.getElementById(id);
    if (el) { el.addEventListener('change', aplicar); }
  });
  var boton = d.getElementById('ag-csv');
  if (boton) { boton.addEventListener('click', descargar); }
  aplicar();
})();
</script>
''' % json.dumps(CONTEXTO, separators=(',', ':'))

# include_plotlyjs=True embebe plotly.js en esta misma salida: el panel, la figura y su
# libreria viajan juntos, asi el cuaderno se ve igual exportado a HTML o en nbviewer.
FIGURA_HTML = pio.to_html(fig, include_plotlyjs=True, full_html=False, div_id=DIV_FIGURA)

display(HTML(PANEL_HTML + FIGURA_HTML + PANEL_JS))

In [7]:
def tabla_etiquetas(desde=None, hasta=None, log_x=False, log_y=False, prep='minmax'):
    """Etiqueta de cada circuito para una configuracion, como DataFrame listo para CSV.

    Es el camino reproducible del boton "Descargar etiquetas (CSV)" del panel: mismo
    esquema, mismo orden y las mismas funciones de agrupamiento, para que el archivo que
    baja el navegador y el que escribe el kernel no puedan divergir.
    """
    etiquetas_mes = [str(m) for m in MESES]
    i = etiquetas_mes.index(desde) if desde else 0
    j = etiquetas_mes.index(hasta) if hasta else len(MESES) - 1
    if i > j:
        i, j = j, i

    agrupada, _ = agrupar(tabla_circuitos(i, j), (log_x, log_y), prep)
    return (
        pd.DataFrame({
            # El periodo si va en cada fila: sin el, la etiqueta no significa nada. La escala
            # y el preproceso quedan solo en el nombre del archivo.
            'desde': str(MESES[i].to_timestamp().date()),
            'hasta': str(MESES[j].to_timestamp(how='end').date()),
            'circuito': agrupada['CIRCUITO'],
            'etiqueta': agrupada['grupo'].map(dict(enumerate(NOMBRES_GRUPOS))),
            'num_eventos': agrupada['num_eventos'].astype(int),
            'uiti_acumulado': agrupada['uiti_acumulado'].round(2),
        })
        .sort_values('uiti_acumulado', ascending=False)
        .reset_index(drop=True)
    )


def guardar_etiquetas(destino=None, **config):
    """Escribe tabla_etiquetas(**config) en reports/interpretability/artifacts/."""
    tabla = tabla_etiquetas(**config)
    if destino is None:
        # La escala y el preproceso ya no son columnas, asi que el nombre del archivo es
        # el unico lugar donde queda registrado en que espacio se corrio K-Means.
        fila = tabla.iloc[0]
        log_x, log_y = config.get('log_x', False), config.get('log_y', False)
        destino = (REPO_ROOT / 'reports' / 'interpretability' / 'artifacts' /
                   f'etiquetas_circuitos_{fila["desde"][:7]}_{fila["hasta"][:7]}'
                   f'_x{"log" if log_x else "lin"}_y{"log" if log_y else "lin"}'
                   f'_{config.get("prep", "minmax")}.csv')
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(destino, index=False)
    return destino


# Por defecto exporta la misma configuracion con que arranca la figura.
etiquetas = tabla_etiquetas()
ruta_csv = guardar_etiquetas()
print(f'{len(etiquetas)} circuitos -> {ruta_csv.relative_to(REPO_ROOT)}')
print(etiquetas['etiqueta'].value_counts().reindex(NOMBRES_GRUPOS).to_string())
etiquetas.head()

208 circuitos -> reports/interpretability/artifacts/etiquetas_circuitos_2025-11_2026-04_xlin_ylin_minmax.csv
etiqueta
Bajo          112
Medio          58
Medio-Alto     21
Alto           17


,desde,hasta,circuito,etiqueta,num_eventos,uiti_acumulado
0,2025-11-01,2026-04-30,BQE23L12,Alto,36,708703.87
1,2025-11-01,2026-04-30,AZA23L17,Alto,9,654633.60
2,2025-11-01,2026-04-30,DON23L12,Alto,16,635570.74
3,2025-11-01,2026-04-30,CHA23L14,Alto,74,549468.07
4,2025-11-01,2026-04-30,HER23L16,Alto,130,507228.42


## Como leerlo

- Un punto es un **circuito**, no un vano: el eje x es cuantos eventos registro en el periodo
  elegido y el eje y cuanto UITI acumulo en ese mismo periodo.
- Las **regiones sombreadas** son la particion del plano, no un contorno de densidad: marcan
  que grupo le tocaria a un circuito segun donde caiga. Son celdas de Voronoi de los cuatro
  centroides, dibujadas en el espacio ajustado, asi que sus fronteras se mueven al cambiar
  cualquiera de los dos logs o el preproceso.
- Los **violines** muestran la distribucion completa de cada variable dentro de cada grupo,
  con su caja y su mediana. Son la contraparte por grupo de los KDE marginales: aquellos
  proyectan todos los grupos sobre un mismo eje, estos los separan.
- El **diagrama de barras** cuenta cuantos circuitos quedaron en cada grupo. Con las dos
  escalas lineales el reparto es muy desbalanceado; con las dos en log se empareja bastante.
- Los dos logs son **independientes a proposito**: las dos variables no tienen la misma forma,
  y transformar solo una es una lectura legitima. Ninguna de las cuatro combinaciones es la
  correcta por defecto.
- El nombre del grupo (`Bajo` a `Alto`) es un ranking **relativo al periodo y al espacio
  elegidos**. Al mover cualquiera de los cuatro controles, K-Means vuelve a particionar y un
  circuito puede cambiar de grupo: no es una etiqueta fija del circuito.

**Limitacion.** `k=4` es una decision operativa (cuatro niveles de riesgo para priorizar
mantenimiento), no un valor que estos dos features impongan. Este cuaderno no valida `k`; se
limita a mostrar la particion y a nombrarla de forma consistente.

---

# Segundo tablero: agrupamiento a nivel de vano

Mismo procedimiento y mismas visualizaciones que el tablero de circuitos, pero cambiando la
unidad: aca cada punto es un **vano** (`FID_VANO`), no un circuito. Son 27.390 vanos contra 208
circuitos, asi que todo lo anterior se repite dos ordenes de magnitud mas denso.

> **Un rango corto vuelve el agrupamiento degenerado.** Sobre el rango completo la mediana es de
> 3 eventos por vano. Sobre **un solo mes**, apenas 10.089 de los 27.390 vanos registran algun
> evento y el **55,8% de esos tiene exactamente uno**, con mediana 1. El eje de eventos se
> aplasta contra el valor 1 y K-Means termina cortando casi solo por UITI. Los rangos cortos
> siguen disponibles, pero conviene leerlos sabiendo esto; el panel avisa cuantos vanos entraron.

> **Como entra tanto dato sin inflar el cuaderno.** Replicar el esquema del primer tablero
> (21 rangos x 8 espacios de coordenadas y etiquetas) pesaria unos 20 MB. En su lugar viaja una
> matriz **vano x mes** de 1,2 MB y el navegador suma los meses del rango elegido, que para una
> suma y un conteo da exactamente lo mismo. Los grupos tampoco viajan: se derivan de los
> centroides con la regla de centroide mas cercano, la misma que dibuja los contornos y que el
> cuaderno verifica contra las etiquetas de scikit-learn.

In [8]:
from scipy.stats import gaussian_kde as _gaussian_kde_control

# --- matriz vano x mes -------------------------------------------------------------------
# El JS suma los meses del rango elegido. Para un conteo y una suma eso es exacto, y evita
# mandar 21 copias de las coordenadas de 27.390 vanos.
_vano = pd.read_csv(REPO_ROOT / 'data' / 'Indicadores_vano_v3.csv',
                    usecols=['CIRCUITO', 'FID_VANO', 'UITI_VANO', 'FECHA'])
_vano['FECHA'] = pd.to_datetime(_vano['FECHA'], errors='coerce')
_vano['UITI_VANO'] = pd.to_numeric(_vano['UITI_VANO'], errors='coerce').fillna(0.0)
# FID_VANO llega numerico con sufijo '.0' inconsistente entre filas; se normaliza como string
# igual que `chec_local_interpreter.plotting._norm_map_id`, para no duplicar vanos por formato.
_vano['FID_VANO'] = (_vano['FID_VANO'].astype('string').str.strip()
                     .str.replace(r'\.0$', '', regex=True))
_vano['MES'] = _vano['FECHA'].dt.to_period('M')

VANOS = sorted(_vano['FID_VANO'].unique())
CIRCUITO_DE_VANO = (_vano.drop_duplicates('FID_VANO').set_index('FID_VANO')['CIRCUITO']
                    .reindex(VANOS).tolist())

_n_mes = (_vano.pivot_table(index='FID_VANO', columns='MES', values='UITI_VANO', aggfunc='count')
          .reindex(VANOS).reindex(columns=MESES).fillna(0).astype(int))
_u_mes = (_vano.pivot_table(index='FID_VANO', columns='MES', values='UITI_VANO', aggfunc='sum')
          .reindex(VANOS).reindex(columns=MESES).fillna(0.0).round(6))
N_MES, U_MES = _n_mes.to_numpy(), _u_mes.to_numpy()

print(f'{len(VANOS):,} vanos x {len(MESES)} meses | '
      f'{len(_vano):,} eventos | {_vano["CIRCUITO"].nunique()} circuitos')
print('eventos por vano en el rango completo -> mediana '
      f'{int(np.median(N_MES.sum(axis=1)))}, con un solo evento '
      f'{int((N_MES.sum(axis=1) == 1).sum()):,}')

27,390 vanos x 6 meses | 159,470 eventos | 208 circuitos
eventos por vano en el rango completo -> mediana 3, con un solo evento 8,934


In [9]:
FRONTERA = []   # (puntos que cambian de grupo por el redondeo, total)


def geometria_vanos(n, u, log_x, log_y, prep):
    """K-Means a 4 grupos sobre los vanos; devuelve solo la geometria de la particion.

    Las etiquetas no se embeben: el JS las deriva de los centroides con la regla de
    centroide mas cercano. Para que eso sea legitimo, la pertenencia que vale en TODOS
    lados -- puntos del mapa, contorno y CSV -- es la que sale de esa misma regla aplicada
    a la geometria YA REDONDEADA que se envia. Asi ningun punto puede quedar pintado del
    lado equivocado de su propia frontera.
    """
    X = np.column_stack([n, u]).astype(float)
    if log_x:
        X[:, 0] = np.log10(X[:, 0])
    if log_y:
        X[:, 1] = np.log10(X[:, 1])

    escalador = PREPROCESOS[prep]().fit(X)
    Z = escalador.transform(X)
    modelo = KMeans(n_clusters=4, random_state=SEMILLA, n_init=10).fit(Z)

    if prep == 'minmax':
        offset, scale = escalador.data_min_, escalador.data_range_
    else:
        offset, scale = escalador.mean_, escalador.scale_
    offset, scale = np.round(offset, 6), np.round(scale, 6)
    centros = np.round(modelo.cluster_centers_, 6)

    def pertenencia(centroides):
        Zr = (X - offset) / scale
        return (((Zr[:, None, :] - centroides[None, :, :]) ** 2).sum(axis=2)).argmin(axis=1)

    cruda = pertenencia(centros)

    # Control contra scikit-learn. No se exige igualdad exacta: un punto que cae justo
    # sobre una frontera de Voronoi puede cruzarla con el redondeo a 1e-6. Lo que no puede
    # pasar es que difieran muchos, que seria una geometria mal armada.
    difieren = int((cruda != modelo.predict(Z)).sum())
    FRONTERA.append((difieren, len(cruda)))
    assert difieren <= max(5, int(0.001 * len(cruda))), (
        f'{difieren} de {len(cruda)} puntos no coinciden con predict(): '
        'eso ya no es un empate de frontera')

    ocupacion = np.bincount(cruda, minlength=4)
    assert ocupacion.min() > 0, f'K-Means dejo un grupo vacio: {ocupacion.tolist()}'

    # El id que devuelve K-Means es arbitrario: el nombre del grupo se asigna por el
    # ranking de la MEDIANA del UITI acumulado, de menor a mayor. Reordenar los centroides
    # no cambia cual es el mas cercano, asi que la pertenencia sigue siendo la misma.
    orden = list(np.argsort([np.median(u[cruda == c]) for c in range(4)]))
    return {
        'logs': [bool(log_x), bool(log_y)],
        'offset': offset.tolist(),
        'scale': scale.tolist(),
        'centroides': centros[orden].tolist(),
    }


# K-Means se ajusta UNA sola vez por espacio, sobre la ventana temporal completa, igual
# que en el tablero de circuitos. Los centroides quedan fijos y cambiar el rango solo
# reevalua a que grupo cae cada vano; las fronteras de Voronoi no se mueven.
_n_full = N_MES.sum(axis=1)
_u_full = U_MES.sum(axis=1)
_full = (_n_full > 0) & (_u_full > 0)
GEOMETRIAS_VANO = {
    str(e): geometria_vanos(_n_full[_full], _u_full[_full], log_x, log_y, prep)
    for e, (log_x, log_y, prep) in enumerate(ESPACIOS)
}

_peor, _tot = max(FRONTERA, key=lambda p: p[0]), sum(p[0] for p in FRONTERA)
print(f'K-Means ajustado {len(GEOMETRIAS_VANO)} veces (una por espacio, sobre la '
      f'ventana completa); los {len(RANGOS)} rangos reusan esos centroides')
print(f'puntos sobre la frontera que el redondeo mueve: {_tot} en total, '
      f'peor caso {_peor[0]} de {_peor[1]:,}')

# Control del KDE que se reimplementa en JavaScript: mismo ancho de banda de Scott que usa
# scipy, comparado sobre un caso real para que las densidades del panel no sean otra cosa.
_m = (N_MES.sum(axis=1) > 0) & (U_MES.sum(axis=1) > 0)
_v = U_MES.sum(axis=1)[_m][:2000]
_grid = np.linspace(_v.min(), _v.max(), 5)
_scipy = _gaussian_kde_control(_v)(_grid)
_bw = np.std(_v, ddof=1) * len(_v) ** (-0.2)
_propio = np.array([np.exp(-0.5 * ((g - _v) / _bw) ** 2).sum() / (len(_v) * _bw * np.sqrt(2 * np.pi))
                    for g in _grid])
print(f'KDE propio vs scipy: error relativo maximo '
      f'{np.max(np.abs(_propio - _scipy) / _scipy):.2e}')

K-Means ajustado 8 veces (una por espacio, sobre la ventana completa); los 21 rangos reusan esos centroides
puntos sobre la frontera que el redondeo mueve: 0 en total, peor caso 0 de 27,390
KDE propio vs scipy: error relativo maximo 3.53e-14


In [10]:
# Misma estructura de trazas que el tablero de circuitos: 1 contorno + 4 scatter +
# 4 KDE del eje x + 4 KDE del eje y + 1 barra + 8 violines.
fig_vano = make_subplots(
    rows=4, cols=2,
    row_heights=[0.16, 0.42, 0.20, 0.22], column_widths=[0.78, 0.22],
    horizontal_spacing=0.02, vertical_spacing=0.075,
    subplot_titles=('', '', '', '', 'Vanos por grupo', '', 'UITI acumulado por grupo', ''),
)

fig_vano.add_trace(go.Contour(                                   # 0: membresia
    z=[[0, 0], [0, 0]], x=[0, 1], y=[0, 1],
    colorscale=ESCALA_CONTORNO, zmin=-0.5, zmax=3.5, showscale=False,
    opacity=0.28, hoverinfo='skip', line=dict(width=1.2, color='rgba(120,20,20,0.6)'),
    contours=dict(start=-0.5, end=3.5, size=1, coloring='fill'), showlegend=False,
), row=2, col=1)
for g in range(4):                                               # 1-4: scatter
    fig_vano.add_trace(go.Scattergl(
        x=[], y=[], mode='markers', name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g],
        marker=dict(size=4, color=COLORES_GRUPOS[g], opacity=0.6),
        hovertext=[], hovertemplate='%{hovertext}<extra></extra>',
    ), row=2, col=1)
for g in range(4):                                               # 5-8: KDE eje x
    fig_vano.add_trace(go.Scatter(
        x=[], y=[], mode='lines', fill='tozeroy', name=NOMBRES_GRUPOS[g],
        legendgroup=NOMBRES_GRUPOS[g], showlegend=False, hoverinfo='skip',
        line=dict(width=1.2, color=COLORES_GRUPOS[g]),
        fillcolor=COLORES_GRUPOS[g].replace('rgb', 'rgba').replace(')', ',0.25)'),
    ), row=1, col=1)
for g in range(4):                                               # 9-12: KDE eje y
    fig_vano.add_trace(go.Scatter(
        x=[], y=[], mode='lines', fill='tozerox', name=NOMBRES_GRUPOS[g],
        legendgroup=NOMBRES_GRUPOS[g], showlegend=False, hoverinfo='skip',
        line=dict(width=1.2, color=COLORES_GRUPOS[g]),
        fillcolor=COLORES_GRUPOS[g].replace('rgb', 'rgba').replace(')', ',0.25)'),
    ), row=2, col=2)
fig_vano.add_trace(go.Bar(                                       # 13: conteo
    x=NOMBRES_GRUPOS, y=[0] * 4, text=[0] * 4, textposition='outside',
    marker=dict(color=COLORES_GRUPOS, line=dict(width=0.5, color='rgba(60,10,10,0.6)')),
    showlegend=False, cliponaxis=False,
    hovertemplate='%{x}: %{y} vanos<extra></extra>',
), row=3, col=1)
# El porcentaje va en una traza aparte, como en el tablero de circuitos: una barra tiene un
# solo par `text`/`textposition`, asi que el conteo de afuera y el porcentaje de adentro no
# pueden salir de la misma. El texto se ancla a media altura de cada barra.
fig_vano.add_trace(go.Scatter(                                   # 14: porcentaje adentro
    x=NOMBRES_GRUPOS, y=[0] * 4, mode='text', text=[''] * 4,
    textposition='middle center',
    textfont=dict(size=11, color=['rgb(40,10,12)', 'rgb(40,10,12)', 'white', 'white']),
    showlegend=False, hoverinfo='skip',
), row=3, col=1)
for fila, etiqueta in [(4, 'UITI acumulado')]:                   # 14-17: violines UITI
    for g in range(4):
        fig_vano.add_trace(go.Violin(
            x=[], y=[], name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g],
            showlegend=False, line=dict(color='rgba(90,15,20,0.85)', width=1),
            fillcolor=COLORES_GRUPOS[g], opacity=0.85,
            box_visible=True, meanline_visible=False, points=False, spanmode='hard',
            hovertemplate=f'%{{x}} -- {etiqueta}: %{{y:,.1f}}<extra></extra>',
        ), row=fila, col=1)

fig_vano.update_layout(
    title=dict(
        text='Agrupamiento de vanos por UITI acumulado y numero de eventos'
             '<br><sup>K-Means (k=4) sobre 27.390 vanos; grupos nombrados por el ranking de la '
             'mediana del UITI acumulado</sup>',
        x=0.5, xanchor='center', yref='container', y=0.97, yanchor='top',
    ),
    legend=dict(title_text='', orientation='h', x=0.5, xanchor='center', y=1.02, yanchor='bottom'),
    margin=dict(t=120, r=30, b=60, l=90),
    height=1180, width=860, template='plotly_white', bargap=0.45, violingap=0.3,
)
fig_vano.update_xaxes(matches='x3', row=1, col=1)
fig_vano.update_yaxes(matches='y3', row=2, col=2)
fig_vano.update_xaxes(title_text='Numero de eventos por vano', row=2, col=1)
fig_vano.update_yaxes(title_text='UITI acumulado por vano', row=2, col=1)
fig_vano.update_yaxes(title_text='Densidad', showticklabels=False, row=1, col=1)
fig_vano.update_xaxes(title_text='Densidad', showticklabels=False, row=2, col=2)
fig_vano.update_yaxes(title_text='Vanos', rangemode='tozero', row=3, col=1)
fig_vano.update_yaxes(title_text='UITI acumulado', row=4, col=1)
for _celda in [(1, 2), (3, 2), (4, 2)]:
    fig_vano.update_xaxes(visible=False, row=_celda[0], col=_celda[1])
    fig_vano.update_yaxes(visible=False, row=_celda[0], col=_celda[1])
for _anotacion in fig_vano.layout.annotations:
    _anotacion.font.size = 12


def _clave_eje_vano(traza, cual):
    # OJO: `traza.y` son los DATOS; la referencia de eje vive en `traza.yaxis` ('y', 'y3'...).
    ref = getattr(traza, f'{cual}axis') or cual
    return f'{cual}axis' + ref[1:]


# Indices declarados una sola vez y enviados al JS, para que insertar una traza no deje
# todos los restyle escribiendo en la equivocada sin que nada falle a la vista.
IDX_VANO = {'contorno': 0, 'mapa': [1, 2, 3, 4], 'kdeX': [5, 6, 7, 8],
            'kdeY': [9, 10, 11, 12], 'barras': 13, 'pct': 14,
            'violinUiti': [15, 16, 17, 18]}
assert len(fig_vano.data) == 19
assert fig_vano.data[IDX_VANO['contorno']].type == 'contour'
assert all(fig_vano.data[i].type == 'scattergl' for i in IDX_VANO['mapa'])
assert all(fig_vano.data[i].type == 'violin' for i in IDX_VANO['violinUiti'])
assert fig_vano.data[IDX_VANO['barras']].type == 'bar'
assert fig_vano.data[IDX_VANO['pct']].mode == 'text'

EJES_VANO = {
    'mapaX': _clave_eje_vano(fig_vano.data[IDX_VANO['mapa'][0]], 'x'),
    'mapaY': _clave_eje_vano(fig_vano.data[IDX_VANO['mapa'][0]], 'y'),
    'kdeXeje': _clave_eje_vano(fig_vano.data[IDX_VANO['kdeX'][0]], 'x'),
    'kdeYeje': _clave_eje_vano(fig_vano.data[IDX_VANO['kdeY'][0]], 'y'),
    'barras': _clave_eje_vano(fig_vano.data[IDX_VANO['barras']], 'y'),
    'violinUiti': _clave_eje_vano(fig_vano.data[IDX_VANO['violinUiti'][0]], 'y'),
}
print(f'{len(fig_vano.data)} trazas | ejes: {EJES_VANO}')

19 trazas | ejes: {'mapaX': 'xaxis3', 'mapaY': 'yaxis3', 'kdeXeje': 'xaxis', 'kdeYeje': 'yaxis4', 'barras': 'yaxis5', 'violinUiti': 'yaxis7'}


In [11]:
DIV_VANO = 'agrupamiento-vanos'

CONTEXTO_VANO = {
    'div': DIV_VANO,
    'meses': [str(m) for m in MESES],
    'finMes': [str(m.to_timestamp(how='end').date()) for m in MESES],
    'rangos': RANGOS,
    'espacios': [[bool(lx), bool(ly), prep] for lx, ly, prep in ESPACIOS],
    'grupos': NOMBRES_GRUPOS,
    'vanos': VANOS,
    'circuitos': CIRCUITO_DE_VANO,
    'nMes': N_MES.tolist(),
    'uMes': U_MES.tolist(),
    'geometrias': GEOMETRIAS_VANO,
    'idx': IDX_VANO,
    'ejes': EJES_VANO,
    'resolucion': 80,
    'gridKde': 64,
    'extension': [float(N_MES.sum(axis=1)[_full].min()), float(N_MES.sum(axis=1)[_full].max()),
                  float(U_MES.sum(axis=1)[_full].min()), float(U_MES.sum(axis=1)[_full].max())],
}

PANEL_VANO_HTML = f'''
<div class="panel-agrup">
  <div><label for="va-desde">Desde</label>
       <input type="date" id="va-desde" min="{PRIMER_DIA}" max="{ULTIMO_DIA}" value="{PRIMER_DIA}"></div>
  <div><label for="va-hasta">Hasta</label>
       <input type="date" id="va-hasta" min="{PRIMER_DIA}" max="{ULTIMO_DIA}" value="{ULTIMO_DIA}"></div>
  <div class="grupo-chk">
    <label class="chk"><input type="checkbox" id="va-logx"> Log eje X (eventos)</label>
    <label class="chk"><input type="checkbox" id="va-logy" checked> Log eje Y (UITI)</label>
  </div>
  <div><label for="va-prep">Preproceso</label>
       <select id="va-prep"><option value="minmax">minmax</option>
                            <option value="zscore">z-score</option></select></div>
  <div><button type="button" id="va-csv">Descargar etiquetas (CSV)</button></div>
  <p class="panel-aviso" id="va-aviso"></p>
</div>
'''

PANEL_VANO_JS = '''
<script type="text/javascript">
(function () {
  var CTX = %s;
  var d = document;
  var ESTADO = null;

  function idxMes(v) { var i = CTX.meses.indexOf((v || '').slice(0, 7)); return i < 0 ? null : i; }

  function idxRango(a, b) {
    for (var i = 0; i < CTX.rangos.length; i++) {
      if (CTX.rangos[i][0] === a && CTX.rangos[i][1] === b) return i;
    }
    return null;
  }

  function kde(vals, log, n) {
    // Densidad gaussiana con el ancho de banda de Scott, el mismo que usa scipy por
    // defecto: bw = desvio muestral * n^(-1/5). El cuaderno compara esta formula contra
    // scipy antes de embeberla, para que la curva del panel no sea otra cosa.
    if (vals.length < 3) { return {x: [], y: []}; }
    var base = log ? vals.map(Math.log10) : vals;
    var m = 0, i;
    for (i = 0; i < base.length; i++) { m += base[i]; }
    m /= base.length;
    var s2 = 0;
    for (i = 0; i < base.length; i++) { s2 += (base[i] - m) * (base[i] - m); }
    var sd = Math.sqrt(s2 / (base.length - 1));
    if (!(sd > 0)) { return {x: [], y: []}; }
    var bw = sd * Math.pow(base.length, -0.2);

    var lo = Math.min.apply(null, base), hi = Math.max.apply(null, base);
    var paso = (hi - lo) / (n - 1), ejes = [], dens = [];
    var k = 1 / (base.length * bw * Math.sqrt(2 * Math.PI));
    for (var j = 0; j < n; j++) {
      var g = lo + j * paso, acc = 0;
      for (i = 0; i < base.length; i++) {
        var z = (g - base[i]) / bw;
        acc += Math.exp(-0.5 * z * z);
      }
      ejes.push(log ? Math.pow(10, g) : g);
      dens.push(acc * k);
    }
    return {x: ejes, y: dens};
  }

  function ejeGrilla(min, max, n, log) {
    var out = [], lo = log ? Math.log10(min) : min, hi = log ? Math.log10(max) : max;
    var paso = (hi - lo) / (n - 1);
    for (var i = 0; i < n; i++) {
      var v = lo + i * paso;
      out.push(log ? Math.pow(10, v) : v);
    }
    return out;
  }

  function grupoDe(nx, uy, geo) {
    var vx = geo.logs[0] ? Math.log10(nx) : nx, vy = geo.logs[1] ? Math.log10(uy) : uy;
    var tx = (vx - geo.offset[0]) / geo.scale[0], ty = (vy - geo.offset[1]) / geo.scale[1];
    var mejor = 0, dmin = Infinity;
    for (var c = 0; c < geo.centroides.length; c++) {
      var a = tx - geo.centroides[c][0], b = ty - geo.centroides[c][1], dd = a * a + b * b;
      if (dd < dmin) { dmin = dd; mejor = c; }
    }
    return mejor;
  }

  function csvActual() {
    var s = ESTADO;
    var out = ['desde,hasta,vano,circuito,etiqueta,num_eventos,uiti_acumulado'];
    var filas = [];
    for (var g = 0; g < 4; g++) {
      for (var i = 0; i < s.idxPorGrupo[g].length; i++) {
        var v = s.idxPorGrupo[g][i];
        filas.push({v: v, g: g, n: s.n[v], u: s.u[v]});
      }
    }
    filas.sort(function (p, q) { return q.u - p.u; });
    filas.forEach(function (f) {
      out.push([s.desde, s.hasta, '"' + CTX.vanos[f.v] + '"', '"' + CTX.circuitos[f.v] + '"',
                CTX.grupos[f.g], f.n, f.u].join(','));
    });
    return out.join('\\n') + '\\n';
  }

  function descargar() {
    if (!ESTADO) { return; }
    var url = URL.createObjectURL(new Blob([csvActual()], {type: 'text/csv;charset=utf-8;'}));
    var a = d.createElement('a');
    a.href = url; a.download = ESTADO.nombre;
    d.body.appendChild(a); a.click(); d.body.removeChild(a);
    setTimeout(function () { URL.revokeObjectURL(url); }, 0);
  }

  function aplicar() {
    var gd = d.getElementById(CTX.div);
    if (!gd || !gd._fullLayout) { return setTimeout(aplicar, 120); }

    var a = idxMes(d.getElementById('va-desde').value);
    var b = idxMes(d.getElementById('va-hasta').value);
    if (a === null) a = 0;
    if (b === null) b = CTX.meses.length - 1;
    if (a > b) { var t = a; a = b; b = t; }
    var logx = d.getElementById('va-logx').checked;
    var logy = d.getElementById('va-logy').checked;
    var prep = d.getElementById('va-prep').value;
    var e = 0;
    for (var i = 0; i < CTX.espacios.length; i++) {
      var esp = CTX.espacios[i];
      if (esp[0] === logx && esp[1] === logy && esp[2] === prep) { e = i; break; }
    }
    // Geometria fija por espacio: el rango de fechas no la cambia.
    var geo = CTX.geometrias[String(e)];
    if (!geo) { return; }

    // Sumar los meses del rango: para un conteo y una suma da exactamente lo mismo que
    // haber agregado el rango entero desde el CSV.
    var n = [], u = [], idxPorGrupo = [[], [], [], []];
    for (i = 0; i < CTX.vanos.length; i++) {
      var sn = 0, su = 0;
      for (var m = a; m <= b; m++) { sn += CTX.nMes[i][m]; su += CTX.uMes[i][m]; }
      n.push(sn); u.push(su);
      // Mismo filtro que Python: sin eventos o sin UITI el punto no existe en escala log.
      if (sn <= 0 || su <= 0) { continue; }
      idxPorGrupo[grupoDe(sn, su, geo)].push(i);
    }
    // Extremos FIJOS, de la ventana completa: los mismos sobre los que se ajustaron los
    // centroides. Recalcularlos por rango reencuadraba los ejes y el contorno.
    var minX = CTX.extension[0], maxX = CTX.extension[1];
    var minY = CTX.extension[2], maxY = CTX.extension[3];

    var mx = [], my = [], mt = [], conteos = [], kx = [], ky = [], vg = [], vy = [];
    for (var g = 0; g < 4; g++) {
      var xs = [], ys = [], ts = [], cats = [];
      for (i = 0; i < idxPorGrupo[g].length; i++) {
        var v = idxPorGrupo[g][i];
        xs.push(n[v]); ys.push(u[v]);
        ts.push('<b>' + CTX.vanos[v] + '</b><br>Circuito: ' + CTX.circuitos[v] +
                '<br>Grupo: <b>' + CTX.grupos[g] + '</b>' +
                '<br>Eventos: ' + n[v] + '<br>UITI acumulado: ' + u[v]);
        cats.push(CTX.grupos[g]);
      }
      mx.push(xs); my.push(ys); mt.push(ts); conteos.push(xs.length);
      vg.push(cats); vy.push(ys);
      var dx = kde(xs, logx, CTX.gridKde);
      kx.push(dx.x); ky.push(dx.y);
    }
    for (g = 0; g < 4; g++) {
      var dy2 = kde(my[g], logy, CTX.gridKde);
      kx.push(dy2.y); ky.push(dy2.x);            // el marginal derecho va transpuesto
    }

    Plotly.restyle(gd, {x: mx, y: my, hovertext: mt}, CTX.idx.mapa);
    Plotly.restyle(gd, {x: kx.slice(0, 4), y: ky.slice(0, 4)}, CTX.idx.kdeX);
    Plotly.restyle(gd, {x: kx.slice(4, 8), y: ky.slice(4, 8)}, CTX.idx.kdeY);
    // El porcentaje se calcula sobre el total dibujado, para que las cuatro cifras sumen
    // 100, y va DENTRO de la barra solo si la barra es lo bastante alta. En una barra muy
    // baja el texto de media altura cae sobre el eje y se encima con el nombre del grupo;
    // en ese caso se pega al conteo de afuera, que es donde si hay sitio.
    var totalV = conteos[0] + conteos[1] + conteos[2] + conteos[3];
    var maxC = Math.max.apply(null, conteos) || 1;
    var pctTxt = conteos.map(function (c) {
      // Simbolo duplicado: el bloque se arma con formateo de cadena.
      return totalV ? (100 * c / totalV).toFixed(1) + '%%' : '';
    });
    var bajo = conteos.map(function (c) { return c / maxC < 0.12; });
    Plotly.restyle(gd, {y: [conteos], text: [conteos.map(
      function (c, i) { return bajo[i] ? c + '  ' + pctTxt[i] : String(c); })]},
      [CTX.idx.barras]);
    Plotly.restyle(gd, {
      y: [conteos.map(function (c) { return c / 2; })],
      text: [pctTxt.map(function (p, i) { return bajo[i] ? '' : p; })],
    }, [CTX.idx.pct]);
    Plotly.restyle(gd, {x: vg, y: vy}, CTX.idx.violinUiti);

    var gx = ejeGrilla(minX, maxX, CTX.resolucion, logx);
    var gy = ejeGrilla(minY, maxY, CTX.resolucion, logy);
    var z = [];
    for (var j = 0; j < gy.length; j++) {
      var fila = [];
      for (i = 0; i < gx.length; i++) { fila.push(grupoDe(gx[i], gy[j], geo)); }
      z.push(fila);
    }
    Plotly.restyle(gd, {z: [z], x: [gx], y: [gy]}, [CTX.idx.contorno]);

    var tx = logx ? 'log' : 'linear', ty = logy ? 'log' : 'linear';
    var cambios = {};
    cambios[CTX.ejes.mapaX + '.type'] = tx;
    cambios[CTX.ejes.mapaY + '.type'] = ty;
    cambios[CTX.ejes.kdeXeje + '.type'] = tx;
    cambios[CTX.ejes.kdeYeje + '.type'] = ty;
    cambios[CTX.ejes.violinUiti + '.type'] = ty;
    cambios[CTX.ejes.barras + '.range'] = [0, Math.max.apply(null, conteos) * 1.18];
    // Limites fijos tambien aca; en log Plotly los espera ya en log10.
    cambios[CTX.ejes.mapaX + '.range'] = logx ? [Math.log10(minX * 0.85), Math.log10(maxX * 1.15)]
                                              : [0, maxX * 1.05];
    cambios[CTX.ejes.mapaY + '.range'] = logy ? [Math.log10(minY * 0.85), Math.log10(maxY * 1.15)]
                                              : [0, maxY * 1.05];
    Plotly.relayout(gd, cambios);

    var total = conteos[0] + conteos[1] + conteos[2] + conteos[3];
    ESTADO = {n: n, u: u, idxPorGrupo: idxPorGrupo,
              desde: CTX.meses[a] + '-01', hasta: CTX.finMes[b],
              nombre: 'etiquetas_vanos_' + CTX.meses[a] + '_' + CTX.meses[b] +
                      '_x' + (logx ? 'log' : 'lin') + '_y' + (logy ? 'log' : 'lin') +
                      '_' + prep + '.csv'};
    d.getElementById('va-aviso').textContent =
      'Rango efectivo: ' + CTX.meses[a] + ' a ' + CTX.meses[b] +
      ' (ajustado a meses completos). ' + total + ' de ' + CTX.vanos.length +
      ' vanos con eventos en el periodo -- reparto ' + conteos.join(' / ') + '.';
  }

  ['va-desde', 'va-hasta', 'va-logx', 'va-logy', 'va-prep'].forEach(function (id) {
    var el = d.getElementById(id);
    if (el) { el.addEventListener('change', aplicar); }
  });
  var boton = d.getElementById('va-csv');
  if (boton) { boton.addEventListener('click', descargar); }
  aplicar();
})();
</script>
''' % json.dumps(CONTEXTO_VANO, separators=(',', ':'))

# include_plotlyjs=False: plotly.js ya viajo con el primer tablero, no hace falta repetirlo.
FIGURA_VANO_HTML = pio.to_html(fig_vano, include_plotlyjs=False, full_html=False,
                               div_id=DIV_VANO)

display(HTML(PANEL_VANO_HTML + FIGURA_VANO_HTML + PANEL_VANO_JS))

In [12]:
def tabla_etiquetas_vano(desde=None, hasta=None, log_x=False, log_y=False, prep='minmax'):
    """Etiqueta de cada vano para una configuracion, como DataFrame listo para CSV.

    Camino reproducible del boton del segundo tablero: mismo esquema y mismo orden.
    """
    etiquetas_mes = [str(m) for m in MESES]
    i = etiquetas_mes.index(desde) if desde else 0
    j = etiquetas_mes.index(hasta) if hasta else len(MESES) - 1
    if i > j:
        i, j = j, i

    n = N_MES[:, i:j + 1].sum(axis=1)
    u = U_MES[:, i:j + 1].sum(axis=1).round(4)
    con_eventos = (n > 0) & (u > 0)
    geo = GEOMETRIAS_VANO[str(ESPACIOS.index((log_x, log_y, prep)))]

    X = np.column_stack([n[con_eventos], u[con_eventos]]).astype(float)
    if log_x:
        X[:, 0] = np.log10(X[:, 0])
    if log_y:
        X[:, 1] = np.log10(X[:, 1])
    Z = (X - np.array(geo['offset'])) / np.array(geo['scale'])
    grupo = (((Z[:, None, :] - np.array(geo['centroides'])[None, :, :]) ** 2)
             .sum(axis=2).argmin(axis=1))

    return (
        pd.DataFrame({
            'desde': str(MESES[i].to_timestamp().date()),
            'hasta': str(MESES[j].to_timestamp(how='end').date()),
            'vano': np.array(VANOS)[con_eventos],
            'circuito': np.array(CIRCUITO_DE_VANO)[con_eventos],
            'etiqueta': [NOMBRES_GRUPOS[g] for g in grupo],
            'num_eventos': n[con_eventos].astype(int),
            'uiti_acumulado': u[con_eventos],
        })
        .sort_values('uiti_acumulado', ascending=False)
        .reset_index(drop=True)
    )


etiquetas_vano = tabla_etiquetas_vano()
_destino = (REPO_ROOT / 'reports' / 'interpretability' / 'artifacts' /
            'etiquetas_vanos_2025-11_2026-04_xlin_ylin_minmax.csv')
_destino.parent.mkdir(parents=True, exist_ok=True)
etiquetas_vano.to_csv(_destino, index=False)
print(f'{len(etiquetas_vano):,} vanos -> {_destino.relative_to(REPO_ROOT)}')
print(etiquetas_vano['etiqueta'].value_counts().reindex(NOMBRES_GRUPOS).to_string())
etiquetas_vano.head()

27,390 vanos -> reports/interpretability/artifacts/etiquetas_vanos_2025-11_2026-04_xlin_ylin_minmax.csv
etiqueta
Bajo          22188
Medio          3723
Medio-Alto      724
Alto            755


,desde,hasta,vano,circuito,etiqueta,num_eventos,uiti_acumulado
0,2025-11-01,2026-04-30,20306252,BQE23L12,Alto,36,40171.8471
1,2025-11-01,2026-04-30,20768787,MTT23L12,Alto,17,28601.1610
2,2025-11-01,2026-04-30,20306265,BQE23L12,Alto,36,27513.6161
3,2025-11-01,2026-04-30,20497969,DON23L12,Alto,22,24457.3216
4,2025-11-01,2026-04-30,20306287,BQE23L12,Alto,18,24267.0750


### Como leerlo

- Un punto es un **vano**, no un circuito. El mismo vano pertenece siempre al mismo circuito,
  que aparece en el tooltip, pero el agrupamiento no lo usa: se decide solo por los eventos y el
  UITI del propio vano.
- Los grupos **no son comparables con los del primer tablero**, aunque compartan nombre. Son dos
  particiones distintas sobre unidades distintas: un vano `Alto` vive casi siempre en un circuito
  `Alto`, pero un circuito `Alto` contiene vanos de los cuatro grupos.
- El **contorno** es la particion del plano por celdas de Voronoi de los cuatro centroides, igual
  que arriba. Las curvas marginales son densidades por grupo calculadas en el navegador con el
  mismo ancho de banda de Scott que usa `scipy`; el cuaderno compara ambas antes de embeberlas.

- La nube sale **rayada en vertical** y la densidad marginal de arriba, con picos: no es un
  artefacto de dibujo. A nivel de vano `num_eventos` es un entero chico (mediana 3), asi que
  todos los vanos con el mismo conteo caen exactamente en la misma abscisa. Un KDE sobre una
  variable discreta suaviza entre valores que no existen; conviene leer esa curva como la altura
  de cada barra entera, no como una densidad continua.

**Limitacion.** Con la mediana en 3 eventos por vano sobre el rango completo, y en 1 sobre un
mes, la coordenada de eventos aporta poca informacion a nivel de vano: buena parte de la
particion termina decidida por el UITI. Es una diferencia real respecto del tablero de circuitos,
donde las dos coordenadas pesan parecido. Se nota en el reparto: en lineal los grupos quedan
22.188 / 3.723 / 724 / 755, y en log-log 5.316 / 8.562 / 8.313 / 5.199.